# Google Search Console Anomaly Detection with Isolation Forest
## Machine Learning Approach for Detecting Unusual Traffic Patterns

This notebook uses Isolation Forest, an unsupervised machine learning algorithm specifically designed for anomaly detection.

**Why Isolation Forest?**
- No complex dependencies
- Fast and efficient
- Works well with time series data
- Doesn't require assumptions about data distribution
- Built into scikit-learn (already available in Colab)

---

## 🚀 Quick Start Instructions

1. **Run Cell 1** (installation) - This fixes numpy compatibility
2. **Restart Runtime** - Click 'Runtime' → 'Restart runtime' in the menu
3. **Skip Cell 1** after restart
4. **Run from Cell 2 onwards** - Execute cells sequentially

## 1. Install Required Packages

In [ ]:
# Fix numpy compatibility issue and install required packages
!pip uninstall -y numpy
!pip install numpy==1.26.4
!pip install -q --upgrade scikit-learn pandas matplotlib seaborn plotly

print("✅ Installation complete!")
print("⚠️  Please click 'Runtime' → 'Restart runtime' before proceeding.")
print("    After restart, skip this cell and start from Cell 2.")

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Check versions
import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"pandas version: {pd.__version__}")
print("✅ All libraries imported successfully!")

## 3. Load Your Google Search Console Data

**Instructions:**
1. Export your GSC data as CSV (Date, Clicks, Impressions, CTR, Position)
2. Run the cell below to upload your file
3. The file should have columns: 'date', 'clicks', 'impressions', 'ctr', 'position'

In [ ]:
from google.colab import files

# Upload your CSV file
print("Please upload your Google Search Console CSV file:")
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"\n✅ File '{filename}' uploaded successfully!")

## 4. Load and Explore Data

In [ ]:
# Load the data
df = pd.read_csv(filename)

print("Raw data loaded. Cleaning and converting...\n")

# Standardize column names (convert to lowercase, remove spaces)
df.columns = df.columns.str.lower().str.strip()

# Convert date column to datetime
date_column = [col for col in df.columns if 'date' in col.lower()][0]
df['date'] = pd.to_datetime(df[date_column])
if date_column != 'date':
    df = df.drop(columns=[date_column])

# Function to clean numeric columns
def clean_numeric(series):
    """Convert strings with commas, percentages to numeric"""
    if series.dtype == 'object':
        # Remove commas and convert
        series = series.astype(str).str.replace(',', '')
        # Handle percentages
        if series.str.contains('%').any():
            series = series.str.rstrip('%').astype(float) / 100
        else:
            series = pd.to_numeric(series, errors='coerce')
    return pd.to_numeric(series, errors='coerce')

# Clean numeric columns
numeric_columns = ['clicks', 'impressions', 'ctr', 'position']
for col in numeric_columns:
    if col in df.columns:
        df[col] = clean_numeric(df[col])

# Remove any rows with missing critical data
df = df.dropna(subset=['date'])

# Sort by date
df = df.sort_values('date').reset_index(drop=True)

# Aggregate by date if there are duplicates
if df['date'].duplicated().any():
    print("⚠️  Found duplicate dates. Aggregating...")
    agg_dict = {col: 'sum' for col in numeric_columns if col in df.columns and col != 'position'}
    if 'position' in df.columns:
        agg_dict['position'] = 'mean'  # Average position
    df = df.groupby('date').agg(agg_dict).reset_index()

# Display basic info
print("✅ Data cleaned and converted!\n")
print("Dataset Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst few rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)

print("\nBasic statistics:")
print(df.describe())

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")
print(f"Total days: {len(df)}")

# Check for any remaining issues
if df.isnull().any().any():
    print("\n⚠️  Warning: Found missing values:")
    print(df.isnull().sum())
    print("\nFilling missing values with forward fill...")
    df = df.ffill().bfill()

## 5. Feature Engineering

We'll create additional features to help the Isolation Forest detect anomalies better.

In [ ]:
# Configuration - Choose primary metric to analyze
PRIMARY_METRIC = 'clicks'  # Options: 'clicks', 'impressions', 'ctr', 'position'

# Create a copy for feature engineering
df_features = df.copy()

# Extract temporal features
df_features['day_of_week'] = df_features['date'].dt.dayofweek  # 0=Monday, 6=Sunday
df_features['day_of_month'] = df_features['date'].dt.day
df_features['month'] = df_features['date'].dt.month
df_features['quarter'] = df_features['date'].dt.quarter
df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)

# Calculate rolling statistics (7-day and 30-day windows)
for metric in ['clicks', 'impressions', 'ctr', 'position']:
    if metric in df_features.columns:
        # Rolling mean
        df_features[f'{metric}_rolling_mean_7'] = df_features[metric].rolling(window=7, min_periods=1).mean()
        df_features[f'{metric}_rolling_mean_30'] = df_features[metric].rolling(window=30, min_periods=1).mean()

        # Rolling std
        df_features[f'{metric}_rolling_std_7'] = df_features[metric].rolling(window=7, min_periods=1).std()
        df_features[f'{metric}_rolling_std_30'] = df_features[metric].rolling(window=30, min_periods=1).std()

        # Deviation from rolling mean
        df_features[f'{metric}_deviation_7'] = (df_features[metric] - df_features[f'{metric}_rolling_mean_7']) / (df_features[f'{metric}_rolling_std_7'] + 1e-5)
        df_features[f'{metric}_deviation_30'] = (df_features[metric] - df_features[f'{metric}_rolling_mean_30']) / (df_features[f'{metric}_rolling_std_30'] + 1e-5)

# Fill any NaN values with forward fill then backward fill
df_features = df_features.ffill().bfill()

print("✅ Feature engineering complete!")
print(f"\nTotal features created: {len(df_features.columns)}")
print(f"\nFeature columns:")
print(df_features.columns.tolist())

## 6. Visualize Raw Data

In [ ]:
# Create visualizations for each metric
metrics = ['clicks', 'impressions', 'ctr', 'position']
available_metrics = [m for m in metrics if m in df.columns]

fig, axes = plt.subplots(len(available_metrics), 1, figsize=(16, 4*len(available_metrics)))

if len(available_metrics) == 1:
    axes = [axes]

for idx, metric in enumerate(available_metrics):
    axes[idx].plot(df['date'], df[metric], linewidth=1.5, color='#2E86AB')
    axes[idx].set_xlabel('Date', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel(metric.capitalize(), fontsize=11, fontweight='bold')
    axes[idx].set_title(f'Google Search Console - {metric.capitalize()} Over Time', fontsize=13, fontweight='bold')
    axes[idx].grid(True, alpha=0.3)
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\n{PRIMARY_METRIC.capitalize()} Statistics:")
print(f"  Mean: {df[PRIMARY_METRIC].mean():.2f}")
print(f"  Std Dev: {df[PRIMARY_METRIC].std():.2f}")
print(f"  Min: {df[PRIMARY_METRIC].min():.2f}")
print(f"  Max: {df[PRIMARY_METRIC].max():.2f}")

## 7. Prepare Features for Isolation Forest

In [ ]:
# Select features for the model
# Exclude date and other non-numeric columns
feature_columns = [col for col in df_features.columns if col != 'date']

# Create feature matrix
X = df_features[feature_columns].values

# Standardize features (important for Isolation Forest)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Features prepared for Isolation Forest")
print(f"Feature matrix shape: {X_scaled.shape}")
print(f"Number of samples: {X_scaled.shape[0]}")
print(f"Number of features: {X_scaled.shape[1]}")

## 8. Train Isolation Forest Model

In [ ]:
# Configuration
CONTAMINATION = 0.05  # Expected proportion of anomalies (5%)
# Adjust this: 0.01 = 1% (very strict), 0.1 = 10% (more lenient)

# Initialize Isolation Forest
iso_forest = IsolationForest(
    n_estimators=100,           # Number of trees in the forest
    contamination=CONTAMINATION, # Expected proportion of outliers
    max_samples='auto',         # Number of samples to draw for each tree
    random_state=42,            # For reproducibility
    n_jobs=-1                   # Use all CPU cores
)

print("Training Isolation Forest model...")
print(f"Expected anomaly rate: {CONTAMINATION*100}%\n")

# Fit the model and predict
predictions = iso_forest.fit_predict(X_scaled)

# Get anomaly scores (lower scores = more anomalous)
anomaly_scores = iso_forest.decision_function(X_scaled)

# Add results to dataframe
df_features['anomaly'] = predictions  # -1 for anomalies, 1 for normal
df_features['anomaly_score'] = anomaly_scores
df_features['is_anomaly'] = df_features['anomaly'] == -1

print("✅ Model trained successfully!")
print(f"\nAnomalies detected: {df_features['is_anomaly'].sum()} out of {len(df_features)} days")
print(f"Anomaly rate: {df_features['is_anomaly'].sum()/len(df_features)*100:.2f}%")

## 9. Classify Anomaly Types

In [ ]:
# Classify anomalies as positive (high values) or negative (low values)
# Based on the primary metric deviation from rolling mean
deviation_col = f'{PRIMARY_METRIC}_deviation_7'

df_features['anomaly_type'] = 'Normal'
df_features.loc[
    (df_features['is_anomaly']) & (df_features[deviation_col] > 0),
    'anomaly_type'
] = 'Positive Anomaly'
df_features.loc[
    (df_features['is_anomaly']) & (df_features[deviation_col] <= 0),
    'anomaly_type'
] = 'Negative Anomaly'

print("Anomaly Classification:")
print(f"  Positive anomalies (unusually high): {(df_features['anomaly_type'] == 'Positive Anomaly').sum()}")
print(f"  Negative anomalies (unusually low): {(df_features['anomaly_type'] == 'Negative Anomaly').sum()}")
print(f"  Normal days: {(df_features['anomaly_type'] == 'Normal').sum()}")

## 10. Visualize Anomalies

In [ ]:
# Main visualization - Primary metric with anomalies
fig, ax = plt.subplots(figsize=(18, 8))

# Plot the primary metric
ax.plot(df_features['date'], df_features[PRIMARY_METRIC],
        label='Actual Values', color='black', linewidth=1.5, zorder=2)

# Plot rolling mean
ax.plot(df_features['date'], df_features[f'{PRIMARY_METRIC}_rolling_mean_7'],
        label='7-day Moving Average', color='#2E86AB', linewidth=2, linestyle='--', alpha=0.7, zorder=1)

# Highlight anomalies
positive_anomalies = df_features[df_features['anomaly_type'] == 'Positive Anomaly']
negative_anomalies = df_features[df_features['anomaly_type'] == 'Negative Anomaly']

ax.scatter(positive_anomalies['date'], positive_anomalies[PRIMARY_METRIC],
          color='#06D6A0', s=150, label='Positive Anomalies',
          zorder=5, edgecolors='black', linewidth=2)

ax.scatter(negative_anomalies['date'], negative_anomalies[PRIMARY_METRIC],
          color='#EF476F', s=150, label='Negative Anomalies',
          zorder=5, edgecolors='black', linewidth=2)

ax.set_xlabel('Date', fontsize=13, fontweight='bold')
ax.set_ylabel(PRIMARY_METRIC.capitalize(), fontsize=13, fontweight='bold')
ax.set_title(f'Anomaly Detection: {PRIMARY_METRIC.capitalize()} (Isolation Forest)',
            fontsize=16, fontweight='bold', pad=20)
ax.legend(loc='best', fontsize=11, framealpha=0.9)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. Anomaly Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Anomaly scores over time
axes[0].plot(df_features['date'], df_features['anomaly_score'], linewidth=1, color='#A23B72')
axes[0].scatter(positive_anomalies['date'], positive_anomalies['anomaly_score'],
               color='#06D6A0', s=80, zorder=5, edgecolors='black')
axes[0].scatter(negative_anomalies['date'], negative_anomalies['anomaly_score'],
               color='#EF476F', s=80, zorder=5, edgecolors='black')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2, label='Decision Boundary')
axes[0].set_xlabel('Date', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Anomaly Score', fontsize=12, fontweight='bold')
axes[0].set_title('Anomaly Scores Over Time\n(Lower = More Anomalous)', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Plot 2: Distribution of anomaly scores
axes[1].hist(df_features[df_features['is_anomaly']==False]['anomaly_score'],
            bins=50, alpha=0.7, label='Normal', color='#2E86AB', edgecolor='black')
axes[1].hist(df_features[df_features['is_anomaly']==True]['anomaly_score'],
            bins=20, alpha=0.7, label='Anomalies', color='#EF476F', edgecolor='black')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Decision Boundary')
axes[1].set_xlabel('Anomaly Score', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Frequency', fontsize=12, fontweight='bold')
axes[1].set_title('Distribution of Anomaly Scores', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nAnomaly Score Statistics:")
print(f"  Mean: {df_features['anomaly_score'].mean():.4f}")
print(f"  Std Dev: {df_features['anomaly_score'].std():.4f}")
print(f"  Min (most anomalous): {df_features['anomaly_score'].min():.4f}")
print(f"  Max (most normal): {df_features['anomaly_score'].max():.4f}")

## 12. Analyze Top Anomalies

In [ ]:
# Get top anomalies sorted by anomaly score (most anomalous first)
anomaly_df = df_features[df_features['is_anomaly']].copy()
anomaly_df = anomaly_df.sort_values('anomaly_score')

print(f"\n{'='*100}")
print(f"TOP 20 ANOMALIES (Most Unusual Days)")
print(f"{'='*100}\n")

# Select relevant columns for display
display_cols = ['date', 'clicks', 'impressions', 'ctr', 'position', 'anomaly_score', 'anomaly_type']
available_display_cols = [col for col in display_cols if col in anomaly_df.columns]

top_anomalies = anomaly_df.head(20)[available_display_cols].copy()

# Format the display
top_anomalies['date'] = top_anomalies['date'].dt.strftime('%Y-%m-%d')
top_anomalies['anomaly_score'] = top_anomalies['anomaly_score'].apply(lambda x: f"{x:.4f}")

# Rename columns for better display
column_names = {
    'date': 'Date',
    'clicks': 'Clicks',
    'impressions': 'Impressions',
    'ctr': 'CTR',
    'position': 'Position',
    'anomaly_score': 'Anomaly Score',
    'anomaly_type': 'Type'
}
top_anomalies = top_anomalies.rename(columns=column_names)

print(top_anomalies.to_string(index=False))

# Summary statistics
print(f"\n{'='*100}")
print(f"ANOMALY SUMMARY")
print(f"{'='*100}")
print(f"Total anomalies detected: {len(anomaly_df)} ({len(anomaly_df)/len(df_features)*100:.2f}% of all days)")
print(f"Positive anomalies: {(anomaly_df['anomaly_type'] == 'Positive Anomaly').sum()}")
print(f"Negative anomalies: {(anomaly_df['anomaly_type'] == 'Negative Anomaly').sum()}")
print(f"\nMost anomalous day: {anomaly_df.iloc[0]['date'].strftime('%Y-%m-%d')} (score: {anomaly_df.iloc[0]['anomaly_score']:.4f})")

# Day of week analysis
print(f"\n{'='*100}")
print(f"ANOMALIES BY DAY OF WEEK")
print(f"{'='*100}")
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
for day_idx, day_name in enumerate(day_names):
    count = (anomaly_df['day_of_week'] == day_idx).sum()
    print(f"{day_name}: {count} anomalies")

## 13. Feature Importance Analysis

Let's see which features contributed most to anomaly detection.

In [ ]:
# Calculate feature importance by looking at how much each feature differs in anomalies vs normal
feature_importance = {}

for col in feature_columns:
    if col in df_features.columns:
        normal_mean = df_features[df_features['is_anomaly']==False][col].mean()
        anomaly_mean = df_features[df_features['is_anomaly']==True][col].mean()

        # Calculate relative difference
        if normal_mean != 0:
            importance = abs((anomaly_mean - normal_mean) / normal_mean)
        else:
            importance = abs(anomaly_mean - normal_mean)

        feature_importance[col] = importance

# Sort by importance
sorted_features = sorted(feature_importance.items(), key=lambda x: x[1], reverse=True)

print("\nTop 15 Most Important Features for Anomaly Detection:")
print(f"{'='*70}")
for i, (feature, importance) in enumerate(sorted_features[:15], 1):
    print(f"{i:2d}. {feature:40s} | Importance: {importance:.4f}")

# Visualize top 10 features
top_10_features = sorted_features[:10]
features_names = [f[0] for f in top_10_features]
features_scores = [f[1] for f in top_10_features]

plt.figure(figsize=(12, 6))
plt.barh(features_names, features_scores, color='#2E86AB', edgecolor='black')
plt.xlabel('Relative Importance', fontsize=12, fontweight='bold')
plt.ylabel('Feature', fontsize=12, fontweight='bold')
plt.title('Top 10 Features Contributing to Anomaly Detection', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 14. Multi-Metric Comparison

In [ ]:
# Compare all metrics during anomaly vs normal days
metrics_to_compare = ['clicks', 'impressions', 'ctr', 'position']
available_metrics = [m for m in metrics_to_compare if m in df_features.columns]

comparison_data = []
for metric in available_metrics:
    normal_mean = df_features[df_features['is_anomaly']==False][metric].mean()
    anomaly_mean = df_features[df_features['is_anomaly']==True][metric].mean()
    percent_diff = ((anomaly_mean - normal_mean) / normal_mean) * 100 if normal_mean != 0 else 0

    comparison_data.append({
        'Metric': metric.capitalize(),
        'Normal Days (Avg)': f"{normal_mean:.2f}",
        'Anomaly Days (Avg)': f"{anomaly_mean:.2f}",
        'Difference (%)': f"{percent_diff:+.2f}%"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nMetric Comparison: Normal Days vs Anomaly Days")
print("="*80)
print(comparison_df.to_string(index=False))

## 15. Export Results

In [ ]:
# Prepare export dataframe with key information
export_cols = ['date', 'clicks', 'impressions', 'ctr', 'position',
               'anomaly_score', 'is_anomaly', 'anomaly_type',
               'day_of_week', 'is_weekend']
available_export_cols = [col for col in export_cols if col in df_features.columns]

export_df = df_features[available_export_cols].copy()

# Add day name
export_df['day_name'] = export_df['date'].dt.day_name()

# Save to CSV
output_filename = f'gsc_anomaly_detection_isolation_forest_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
export_df.to_csv(output_filename, index=False)

print(f"✅ Results exported to: {output_filename}")
print(f"\nDownloading file...")

# Download the file
files.download(output_filename)

print(f"\n📊 Export complete! The file contains:")
print(f"  - All dates with metrics")
print(f"  - Anomaly scores (lower = more anomalous)")
print(f"  - Anomaly flags and classifications")
print(f"  - Temporal information (day of week, weekend flag)")

# Also export just the anomalies
anomalies_filename = f'gsc_anomalies_only_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
export_df[export_df['is_anomaly']].to_csv(anomalies_filename, index=False)
files.download(anomalies_filename)
print(f"\n✅ Anomalies-only file exported to: {anomalies_filename}")

## 16. Optional: Adjust Sensitivity

If you want to detect more or fewer anomalies, adjust the contamination parameter and re-run.

In [ ]:
# Try different contamination values
contamination_values = [0.01, 0.03, 0.05, 0.10, 0.15]

print("Testing different sensitivity levels...\n")
print(f"{'Contamination':<15} {'Anomalies Detected':<20} {'Percentage':<15}")
print("="*50)

for cont in contamination_values:
    model = IsolationForest(contamination=cont, random_state=42, n_jobs=-1)
    preds = model.fit_predict(X_scaled)
    n_anomalies = (preds == -1).sum()
    percentage = (n_anomalies / len(preds)) * 100

    marker = " ← Current" if cont == CONTAMINATION else ""
    print(f"{cont:<15.2f} {n_anomalies:<20} {percentage:<15.2f}%{marker}")

print("\n💡 To change sensitivity, modify CONTAMINATION in Cell 8 and re-run from there.")
print("   - Lower values (0.01) = fewer, more extreme anomalies")
print("   - Higher values (0.10) = more anomalies detected")

## Next Steps and Tips

### Understanding Your Results

1. **Anomaly Score**: Lower (more negative) scores indicate more unusual patterns
2. **Positive Anomalies**: Days with unexpectedly high performance (good news!)
3. **Negative Anomalies**: Days with unexpectedly low performance (investigate these)

### Investigation Checklist

For each detected anomaly, consider:
- **Algorithm Updates**: Google core updates, spam updates
- **Technical Issues**: Site downtime, crawl errors, indexing problems
- **Content Changes**: New content published, old content updated/removed
- **Seasonality**: Holidays, events, seasonal trends
- **Competition**: Competitor actions, SERP changes
- **Marketing**: Campaigns, PR, viral content
- **External Factors**: News events, trending topics

### Adjusting the Model

- **Too many anomalies?** Decrease `CONTAMINATION` (e.g., from 0.05 to 0.02)
- **Too few anomalies?** Increase `CONTAMINATION` (e.g., from 0.05 to 0.10)
- **Change primary metric**: Modify `PRIMARY_METRIC` in Cell 5
- **Add more features**: Include custom business metrics in your CSV

### Automation Ideas

1. Schedule this notebook to run weekly/monthly
2. Set up alerts for new anomalies
3. Create a dashboard with historical anomaly trends
4. Cross-reference with Google Analytics data

## Why Isolation Forest Works Well

- **No assumptions**: Doesn't assume normal distribution
- **Handles multiple features**: Considers all metrics simultaneously
- **Fast**: Efficient even with large datasets
- **Unsupervised**: Doesn't need labeled anomalies for training
- **Robust**: Works with noisy data and missing values

Happy anomaly hunting! 🔍